In [49]:
# STEP 1: Install & Import Libraries
# ============================================

!pip install kagglehub

import kagglehub
import pandas as pd
import numpy as np
import random
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [50]:
# ============================================
# STEP 2: Upload kaggle.json
# ============================================

from google.colab import files
files.upload()   # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"thoratparasmansing","key":"e27fd26addf185ddef8c3feffb1e5ca4"}'}

### How to set up your Kaggle API Key in Colab:

1.  **Generate Kaggle API Token:**
    *   Go to [Kaggle](https://www.kaggle.com/).
    *   Log in to your account.
    *   Click on your profile picture in the top right corner and select 'My Account'.
    *   Scroll down to the 'API' section and click on 'Create New API Token'. This will download a `kaggle.json` file.

2.  **Upload `kaggle.json` to Colab Secrets:**
    *   In Google Colab, click on the '🔑' icon (Secrets) in the left-hand sidebar.
    *   Click 'Add new secret'.
    *   For the 'Name' of the secret, type `KAGGLE_USERNAME` and for the 'Value', open your `kaggle.json` file and copy the value associated with the key `username`.
    *   Add another secret: 'Name' as `KAGGLE_KEY` and 'Value' as the value associated with the key `key` from your `kaggle.json` file.
    *   Enable 'Notebook access' for both secrets.

After setting up these secrets, you can uncomment the line for `drinks_path` in the previous cell and rerun it. `kagglehub` should then be able to use these credentials automatically.

In [51]:
# ============================================
# STEP 3: Setup Kaggle
# ============================================

import os

os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [52]:
# ============================================
# STEP 4: Download Datasets
# ============================================

# Food dataset
!kaggle datasets download -d irkaal/foodcom-recipes-and-reviews
!unzip -q foodcom-recipes-and-reviews.zip -d food_data

# Indian food dataset
!kaggle datasets download -d nehaprabhavalkar/indian-food-101
!unzip -q indian-food-101.zip -d indian_data

# Drinks dataset
!kaggle datasets download -d justinpakzad/drinks-recipes
!unzip -q drinks-recipes.zip -d drinks_data

print("✅ Datasets downloaded")

Dataset URL: https://www.kaggle.com/datasets/irkaal/foodcom-recipes-and-reviews
License(s): CC0-1.0
foodcom-recipes-and-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)
replace food_data/recipes.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: No
Dataset URL: https://www.kaggle.com/datasets/nehaprabhavalkar/indian-food-101
License(s): copyright-authors
indian-food-101.zip: Skipping, found more recently modified local copy (use --force to force download)
replace indian_data/indian_food.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: No
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open drinks-recipes.zip, drinks-recipes.zip.zip or drinks-recipes.zip.ZIP.
✅ Datasets downloaded


In [53]:
# ============================================
# STEP 4: PREPARE COMBINED DATA
# ============================================
# The 'data' variable in the kernel state contains the combined features and target.
# Initialize df directly from this data to ensure consistency for model training.
df = pd.DataFrame(data)

# Fill missing values (safe handling)
# Using ffill after creating the DataFrame from 'data' which should be complete.
df = df.ffill()

In [54]:
# ============================================
#STEP 5: ENCODE CATEGORICAL DATA (NO STRUCTURE CHANGE)
# ============================================

label_encoders = {}

# Get list of object (string) columns from df
categorical_cols = df.select_dtypes(include='object').columns

print(f"Encoding categorical columns: {list(categorical_cols)}")

for col in categorical_cols:
    le = LabelEncoder()
    # Fit and transform the column, ensuring it's treated as string
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("✅ Categorical data encoded.")
print(f"Label encoders created for: {list(label_encoders.keys())}")

Encoding categorical columns: ['Mood', 'Weather', 'Time', 'Food', 'Drink']
✅ Categorical data encoded.
Label encoders created for: ['Mood', 'Weather', 'Time', 'Food', 'Drink']


In [55]:
# ============================================
# STEP 6: DEFINE FEATURES & TARGET
# ============================================
# ⚠️ IMPORTANT: change 'target' to your actual column if needed

target_col = 'Food'   # Changed to 'Food' for food prediction

X = df.drop(target_col, axis=1)
y = df[target_col]


In [56]:
# ============================================
# STEP 7: TRAIN-TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [57]:
# ============================================
# STEP 8: FEATURE SCALING (IMPORTANT)
# ============================================
from sklearn.preprocessing import StandardScaler
import pandas as pd

# The error 'ValueError: could not convert string to float: 'Excited''
# indicates that X_train contains non-numeric (string) data when StandardScaler expects numbers.
# This often happens when categorical features are not fully encoded to numerical values.

# Based on the kernel state of X_train, columns 'Mood', 'Weather', 'Time', and 'Food'
# are still in string format. We need to convert them using the label_encoders
# that were generated in a previous step.

# Identify categorical columns that are still objects in X_train
categorical_cols_to_encode = [col for col in X_train.columns if X_train[col].dtype == 'object']

for col in categorical_cols_to_encode:
    if col in label_encoders:
        # Apply the pre-fitted LabelEncoder to transform string values to numerical
        # .astype(str) is used to ensure the input to transform is string, preventing errors
        # if a column accidentally became mixed type.
        X_train[col] = label_encoders[col].transform(X_train[col].astype(str))
        X_test[col] = label_encoders[col].transform(X_test[col].astype(str))
    else:
        # If for some reason a LabelEncoder is not found for an object column,
        # try a robust conversion to numeric, filling NaNs (e.g., if it was meant
        # to be numeric but became object due to mixed types).
        print(f"Warning: No LabelEncoder found for column '{col}'. Attempting direct numeric conversion.")
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

# Now that all features in X_train and X_test should be numeric, proceed with scaling.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [58]:
# ============================================
# STEP 9: INITIALIZE MODELS
# ============================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "SVM": SVC(probability=True)
}

In [59]:
# ============================================
# STEP 10: TRAIN MODELS
# ============================================
for name, model in models.items():
    model.fit(X_train, y_train)



In [60]:
# ============================================
# STEP 11: MODEL ACCURACY
# ============================================
from sklearn.metrics import accuracy_score

print("\n📊 Model Accuracy:\n")

for name, model in models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"{name}: {acc:.2f}")


📊 Model Accuracy:

KNN: 0.00
Decision Tree: 0.00
Random Forest: 0.00
SVM: 0.00


In [61]:
# ============================================
# STEP 12: PREDICTION FUNCTION
# ============================================
def predict_and_suggest(input_data):
    input_data = np.array(input_data).reshape(1, -1)
    input_data = scaler.transform(input_data)

    results = {}
    vote_score = {}

    for name, model in models.items():
        probs = model.predict_proba(input_data)[0]
        pred = np.argmax(probs)
        confidence = np.max(probs)

        results[name] = {
            "prediction": pred,
            "confidence": confidence
        }

        # weighted voting
        if pred not in vote_score:
            vote_score[pred] = 0
        vote_score[pred] += confidence

    final_prediction = max(vote_score, key=vote_score.get)

    return results, final_prediction

In [62]:
# ============================================
# STEP 13.1: INTERACTIVE PREDICTION FUNCTION
# ============================================

def get_interactive_food_suggestion():
    print("\n--- Get Your Personalized Food Suggestion ---")

    # Get user input for Mood
    print("\nAvailable Moods:", ', '.join([le_mood.replace("'", "") for le_mood in label_encoders['Mood'].classes_]))
    mood_input = input("Enter your current mood: ")
    while mood_input not in label_encoders['Mood'].classes_:
        print("Invalid mood. Please choose from the available moods.")
        mood_input = input("Enter your current mood: ")

    # Get user input for Weather
    print("\nAvailable Weathers:", ', '.join([le_weather.replace("'", "") for le_weather in label_encoders['Weather'].classes_]))
    weather_input = input("Enter the current weather: ")
    while weather_input not in label_encoders['Weather'].classes_:
        print("Invalid weather. Please choose from the available weathers.")
        weather_input = input("Enter the current weather: ")

    # Get user input for Time
    print("\nAvailable Times:", ', '.join([le_time.replace("'", "") for le_time in label_encoders['Time'].classes_]))
    time_input = input("Enter the time of day (Morning, Afternoon, Evening, Night): ")
    while time_input not in label_encoders['Time'].classes_:
        print("Invalid time. Please choose from the available times.")
        time_input = input("Enter the time of day: ")

    # Suggest a drink based on mood, weather, and time
    def suggest_drink(mood, weather, time):
        drink_suggestions = []

        # Mood-based suggestions
        if mood in ['Happy', 'Excited']:
            drink_suggestions.extend(['Juice', 'Smoothie', 'Lemonade'])
        elif mood in ['Relaxed', 'Bored']:
            drink_suggestions.extend(['Tea', 'Coffee', 'Coconut Water'])
        elif mood in ['Sad', 'Stressed']:
            drink_suggestions.extend(['Hot Chocolate', 'Milkshake'])
        elif mood == 'Angry':
            drink_suggestions.extend(['Cold Coffee', 'Lemonade'])

        # Weather-based suggestions
        if weather == 'Hot':
            drink_suggestions.extend(['Cold Coffee', 'Juice', 'Lemonade', 'Coconut Water', 'Smoothie', 'Lassi'])
        elif weather == 'Cold':
            drink_suggestions.extend(['Hot Chocolate', 'Masala Chai', 'Coffee', 'Tea'])
        elif weather == 'Rainy':
            drink_suggestions.extend(['Masala Chai', 'Tea', 'Coffee', 'Hot Chocolate'])
        elif weather == 'Windy':
            drink_suggestions.extend(['Tea', 'Coffee'])
        elif weather == 'Humid':
            drink_suggestions.extend(['Juice', 'Lemonade', 'Coconut Water'])

        # Time-based suggestions
        if time == 'Morning':
            drink_suggestions.extend(['Coffee', 'Tea', 'Juice', 'Masala Chai', 'Smoothie'])
        elif time == 'Afternoon':
            drink_suggestions.extend(['Coffee', 'Juice', 'Smoothie', 'Lassi', 'Cold Coffee'])
        elif time == 'Evening':
            drink_suggestions.extend(['Tea', 'Milkshake', 'Hot Chocolate'])
        elif time == 'Night':
            drink_suggestions.extend(['Hot Chocolate', 'Tea', 'Milkshake'])

        # Filter suggestions to only include drinks available in label_encoders['Drink'].classes_
        available_drinks = list(label_encoders['Drink'].classes_)
        valid_suggestions = [d for d in drink_suggestions if d in available_drinks]

        if valid_suggestions:
            # Pick a random drink from the valid suggestions
            return random.choice(valid_suggestions)
        else:
            # Fallback: if no specific suggestion, pick a random available drink
            return random.choice(available_drinks)

    drink_suggestion = suggest_drink(mood_input, weather_input, time_input)
    print(f"\nI suggest you try a: {drink_suggestion}")
    drink_input = drink_suggestion # Use the suggested drink as input

    # Encode user inputs using the fitted LabelEncoders
    encoded_mood = label_encoders['Mood'].transform([mood_input])[0]
    encoded_weather = label_encoders['Weather'].transform([weather_input])[0]
    encoded_time = label_encoders['Time'].transform([time_input])[0]
    encoded_drink = label_encoders['Drink'].transform([drink_input])[0]

    # Assemble the input data in the correct order as per X columns: ['Mood', 'Weather', 'Time', 'Drink']
    user_input_encoded = np.array([
        encoded_mood,
        encoded_weather,
        encoded_time,
        encoded_drink
    ]).reshape(1, -1)

    # Get prediction
    results, final_pred_encoded = predict_and_suggest(user_input_encoded)

    # Decode the final prediction
    final_food_suggestion = label_encoders['Food'].inverse_transform([final_pred_encoded])[0]

    print(f"\n\n✅ Your personalized food suggestion: {final_food_suggestion}")

    print("\n🔍 Individual Model Results:")
    for model_name, res in results.items():
        decoded_pred = label_encoders['Food'].inverse_transform([res['prediction']])[0]
        print(f"  {model_name}: Pred = {decoded_pred}, Confidence = {res['confidence']:.2f}")


### Try it out! Run the cell below to get an interactive food suggestion.

In [63]:
# Call the interactive function to get a food suggestion
get_interactive_food_suggestion()


--- Get Your Personalized Food Suggestion ---

Available Moods: Angry, Bored, Excited, Happy, Relaxed, Sad, Stressed
Enter your current mood: Relaxed

Available Weathers: Cold, Hot, Humid, Rainy, Windy
Enter the current weather: Hot

Available Times: Afternoon, Evening, Morning, Night
Enter the time of day (Morning, Afternoon, Evening, Night): Night

I suggest you try a: Cold Coffee


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(




✅ Your personalized food suggestion: Apple Pie Martini

🔍 Individual Model Results:
  KNN: Pred = Apple Pie Martini, Confidence = 0.20
  Decision Tree: Pred = Apple Pie Martini, Confidence = 0.33
  Random Forest: Pred = Apple Pie Martini, Confidence = 0.33
  SVM: Pred = Japanese Mum's Chicken, Confidence = 0.00


In [64]:
# ============================================
# STEP 13: TEST WITH SAMPLE INPUT
# ============================================
# Get the raw sample input as a pandas Series
raw_sample_input_series = X.iloc[0].copy()

# The sample input is already numerically encoded because X was derived from the df
# that was fully encoded in STEP 5. No further encoding is needed here.
# Convert the numerically encoded Series to a NumPy array and reshape for the model
sample_input_encoded = raw_sample_input_series.values.reshape(1, -1)

results, final_pred = predict_and_suggest(sample_input_encoded)

# Decode the final prediction back to the original food name
predicted_food_name = label_encoders[target_col].inverse_transform([final_pred])[0]

print("\n🔍 Individual Model Results:\n")
for model in results:
    # Decode individual model predictions for clarity
    decoded_pred = label_encoders[target_col].inverse_transform([results[model]['prediction']])[0]
    print(f"{model}: Pred = {decoded_pred}, Confidence = {results[model]['confidence']:.2f}")

print(f"\n✅ FINAL SUGGESTION: {predicted_food_name}")


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



🔍 Individual Model Results:

KNN: Pred = Chocolate Chunk Cookies (Low Fat/Low Calorie), Confidence = 0.20
Decision Tree: Pred = Chocolate Chunk Cookies (Low Fat/Low Calorie), Confidence = 0.20
Random Forest: Pred = Pork in Hot Peanut Sauce, Confidence = 0.21
SVM: Pred = Japanese Mum's Chicken, Confidence = 0.00

✅ FINAL SUGGESTION: Chocolate Chunk Cookies (Low Fat/Low Calorie)
